In [1]:
import os
import re
import json
import pandas as pd

In [2]:
# Define os caminhos
JSON_PATH = "../corpus.json"
CSV_PATH  = "authors.csv"

In [3]:
# Carrega ou cria o DataFrame com os dados dos autores
if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH, dtype=str)
else:
    df = pd.DataFrame(columns=["titulo_original","titulo_revisado","autor","afiliação","orcid"])

#Carrega o JSON local
with open(JSON_PATH, encoding="utf-8") as f:
    corpus = json.load(f)


In [4]:
def parse_autores(raw: str):
    # 1) separa autores x afiliações
    m = re.search(r"\b\d+\s", raw)
    if not m:
        return []
    split_idx    = m.start()
    authors_part = raw[:split_idx].strip()
    aff_part     = raw[split_idx:].strip()

    # 2) constrói mapa índice→afiliação COM A NOVA REGEX
    aff_map = {
        idx: text.strip()
        for idx, text in re.findall(
            r"(\d+)\s+(.+?)(?=\s+\d+\s+|$)",
            aff_part,
            flags=re.DOTALL
        )
    }

    # 3) captura cada autor + índices (1 ou vários)
    pattern = re.compile(
        r"""(?P<nome>
               [A-ZÀ-ÖØ-Ý][\w\.\-ÇçãõÃÕéÉíÍóÓúÚâÂêÊîÎôÔûÛ\s]+?
            )\s*
            (?P<idxs>
               (?:\[\d+\]
                  (?:\s*\[,\]\s*\[\d+\])*
               )+
            )
        """,
        re.VERBOSE,
    )

    results = []
    for m in pattern.finditer(authors_part):
        nome = m.group("nome").strip()
        idxs = re.findall(r"\[(\d+)\]", m.group("idxs"))
        afs  = [aff_map[i] for i in idxs if i in aff_map]
        if afs:
            results.append((nome, afs))
    return results

In [5]:
for art in corpus:
    titulo     = art.get("titulo","").strip()
    titulo_rev = titulo
    raw        = art.get("autores","")
    for nome, afs in parse_autores(raw):
        # ignora linhas “corrompidas”
        if "**" in nome:
            continue
        af_str = ", ".join(afs)
        exists = ((df["titulo_original"] == titulo) &
                  (df["autor"]           == nome)).any()
        if not exists:
            df.loc[len(df)] = [titulo, titulo_rev, nome, af_str, ""]

In [6]:

def parse_autores(raw: str):
    # 1) separa autores x afiliações
    m = re.search(r"\b\d+\s", raw)
    if not m:
        return []
    split_idx    = m.start()
    authors_part = raw[:split_idx].strip()
    aff_part     = raw[split_idx:].strip()

    # 2) constrói mapa índice→afiliação COM A NOVA REGEX
    aff_map = {
        idx: text.strip()
        for idx, text in re.findall(
            r"(\d+)\s+(.+?)(?=\s+\d+\s+|$)",
            aff_part,
            flags=re.DOTALL
        )
    }

    # 3) captura cada autor + índices (1 ou vários)
    pattern = re.compile(
        r"""(?P<nome>
               [A-ZÀ-ÖØ-Ý][\w\.\-ÇçãõÃÕéÉíÍóÓúÚâÂêÊîÎôÔûÛ\s]+?
            )\s*
            (?P<idxs>
               (?:\[\d+\]
                  (?:\s*\[,\]\s*\[\d+\])*
               )+
            )
        """,
        re.VERBOSE,
    )

    results = []
    for m in pattern.finditer(authors_part):
        nome = m.group("nome").strip()
        idxs = re.findall(r"\[(\d+)\]", m.group("idxs"))
        afs  = [aff_map[i] for i in idxs if i in aff_map]
        if afs:
            results.append((nome, afs))
    return results

# 1) monta ou carrega DataFrame
if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH, dtype=str)
else:
    df = pd.DataFrame(columns=["titulo_original","titulo_revisado","autor","afiliação","orcid"])

# 2) carrega JSON local
with open(JSON_PATH, encoding="utf-8") as f:
    corpus = json.load(f)

# 3) percorre e adiciona novos
for art in corpus:
    titulo     = art.get("titulo","").strip()
    titulo_rev = titulo
    raw        = art.get("autores","")
    for nome, afs in parse_autores(raw):
        # ignora linhas “corrompidas”
        if "**" in nome:
            continue
        af_str = ", ".join(afs)
        exists = ((df["titulo_original"] == titulo) &
                  (df["autor"]           == nome)).any()
        if not exists:
            df.loc[len(df)] = [titulo, titulo_rev, nome, af_str, ""]

In [7]:
df.tail()

,titulo_original,titulo_revisado,autor,afiliação,orcid
63,Impact of Shot Noise Estimation on the Secret ...,Impact of Shot Noise Estimation on the Secret ...,Armando N. Pinto,NaN,NaN
64,Desafios e Oportunidades de Pesquisa para o Ro...,Desafios e Oportunidades de Pesquisa para o Ro...,Antônio Abelém,Universidade Federal do Pará (UFPA),NaN
65,Desafios e Oportunidades de Pesquisa para o Ro...,Desafios e Oportunidades de Pesquisa para o Ro...,Christian R. Esteve Rothenberg,Universidade Estadual de Campinas (Unicamp),NaN
66,Distributed Quantum Walk Control Plane Impleme...,Distributed Quantum Walk Control Plane Impleme...,Don Towsley,Manning College of Information and Computer Sc...,NaN
67,Demonstration of a PolarizationDemonstration o...,Demonstration of a PolarizationDemonstration o...,Nelson J. Muga,"Instituto de Telecomunicações – Aveiro Aveiro,...",NaN


In [8]:
df.to_csv(CSV_PATH, index=False, encoding="utf-8")


In [9]:
contagem_autores= df.groupby("autor").size().reset_index(name="quantidade").sort_values(by="quantidade", ascending=False)
contagem_autores

,autor,quantidade
6,Antônio Abelém,6
16,Diego Abreu,6
37,Nuno A. Silva,5
36,Nelson J. Muga,4
12,Arthur Pimentel,4
13,Christian R. Esteve Rothenberg,3
9,Armando N. Pinto,3
14,Daniel Pereira,2
15,David Tavares,2
1,"**Sara T. Mantey [1,2]",1


In [10]:
len(contagem_autores)

42

### Update ID Autores revisados

In [1]:
import pandas as pd
import json
import os

# Arquivos
CSV_AUTORES = "autor_revisado.csv"
CSV_links = "artigo_link.csv"
JSON_CORPUS = "../corpus.json"


df_autores = pd.read_csv(CSV_AUTORES, dtype=str).fillna("")
df_links = pd.read_csv(CSV_links, dtype=str).fillna("")
with open(JSON_CORPUS, encoding="utf-8") as f:
    corpus = json.load(f)

# 2. Cria dicionário titulo → key_file extraída do storage_key
titulo_to_key = {}
for art in corpus:
    titulo = art.get("titulo", "").strip()
    storage_key = art.get("storage_key", "")
    key_file = os.path.splitext(os.path.basename(storage_key))[0] if storage_key else ""
    if titulo and key_file:
        titulo_to_key[titulo] = key_file


df_autores["key_file"] = df_autores["titulo_revisado"].map(lambda t: titulo_to_key.get(t.strip(), ""))
df_links["key_file"] = df_links["titulo"].map(lambda t: titulo_to_key.get(t.strip(), ""))

df_autores.to_csv(CSV_AUTORES, index=False, encoding="utf-8")
df_links.to_csv(CSV_links, index=False, encoding="utf-8")
